In [ ]:
# ==========================================
# TRIAGEGEIST: Multimodal Triage Acuity Prediction
# ==========================================
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
import warnings
import os

warnings.filterwarnings('ignore')

# 1. Dynamically Find Data Path (This fixes the FileNotFoundError)
print("Searching for dataset directory...")
data_dir = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'train.csv' in filenames:
        data_dir = dirname
        break

if data_dir is None:
    raise FileNotFoundError("Dataset not found! Please ensure 'Triagegeist' is attached in the right-hand 'Input' menu.")

print(f"Dataset successfully located at: {data_dir}")

# 2. Load Data
print("Loading data...")
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
chief_complaints = pd.read_csv(os.path.join(data_dir, 'chief_complaints.csv'))
history = pd.read_csv(os.path.join(data_dir, 'patient_history.csv'))
sample_sub = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

# 3. Merge Data
print("Merging datasets...")
def merge_data(df):
    df = df.merge(history, on='patient_id', how='left')
    df = df.merge(chief_complaints, on='patient_id', how='left')
    return df

train = merge_data(train)
test = merge_data(test)

# Save targets and drop from train to match test shape
y_train = train['triage_acuity']
train_ids = train['patient_id']
test_ids = test['patient_id']

# Drop target-leakage columns from train (disposition, ed_los_hours are outcomes)
cols_to_drop = ['triage_acuity', 'disposition', 'ed_los_hours', 'patient_id']
X_train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
X_test = test.drop(columns=['patient_id'])

# 4. Feature Engineering & Handling Clinical Missingness
print("Engineering features...")
vitals = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2']

for df in [X_train, X_test]:
    # Clinical Insight: Create flags for missing vitals
    for vital in vitals:
        df[f'{vital}_is_missing'] = df[vital].isnull().astype(int)
    
    # Replace -1 in pain score with NaN to treat it as missing, then create a flag
    df['pain_score'] = df['pain_score'].replace(-1, np.nan)
    df['pain_score_is_missing'] = df['pain_score'].isnull().astype(int)

# 5. Process Categorical Data
print("Encoding categoricals...")
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
if 'chief_complaint_raw' in cat_cols:
    cat_cols.remove('chief_complaint_raw') 

for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)
    
    le = LabelEncoder()
    le.fit(pd.concat([X_train[col], X_test[col]]))
    X_train[col] = le.transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

# 6. NLP Feature Extraction on Chief Complaints
print("Processing unstructured text via TF-IDF...")
tfidf = TfidfVectorizer(max_features=200, stop_words='english', ngram_range=(1,2))

X_train['chief_complaint_raw'] = X_train['chief_complaint_raw'].fillna('')
X_test['chief_complaint_raw'] = X_test['chief_complaint_raw'].fillna('')

tfidf.fit(X_train['chief_complaint_raw'])
train_tfidf = tfidf.transform(X_train['chief_complaint_raw']).toarray()
test_tfidf = tfidf.transform(X_test['chief_complaint_raw']).toarray()

tfidf_cols = [f'tfidf_{i}' for i in range(train_tfidf.shape[1])]
train_tfidf_df = pd.DataFrame(train_tfidf, columns=tfidf_cols, index=X_train.index)
test_tfidf_df = pd.DataFrame(test_tfidf, columns=tfidf_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=['chief_complaint_raw']), train_tfidf_df], axis=1)
X_test = pd.concat([X_test.drop(columns=['chief_complaint_raw']), test_tfidf_df], axis=1)

# 7. Model Training
print("Training LightGBM model...")
clf = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
    verbose=-1
)

clf.fit(X_train, y_train)

# 8. Prediction & Submission
print("Generating predictions...")
preds = clf.predict(X_test)

submission = pd.DataFrame({
    'patient_id': test_ids,
    'triage_acuity': preds
})

submission.to_csv('submission.csv', index=False)
print("SUCCESS! submission.csv created. You can now submit this to the competition.")